In [1]:
# ==============================
# 第一步：导入所需库
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- 建模相关的库 ---
# train_test_split: 把数据拆成「训练集」和「测试集」
#   训练集用来教模型学习规律，测试集用来考试看模型学得怎么样
#   如果不拆分，用同一批数据训练+考试，模型会"作弊"（过拟合），分数虚高
from sklearn.model_selection import train_test_split

# StandardScaler: 标准化（把不同量纲的特征拉到同一尺度）
#   比如价格范围 0~300，而小时数范围 0~23，不标准化的话
#   模型会误以为价格"更重要"（因为数值更大），标准化后系数才有可比性
from sklearn.preprocessing import StandardScaler

# LogisticRegression: 逻辑回归模型本身
#   虽然名字里有"回归"，但其实是做分类的（预测 0 或 1）
#   它输出的是一个 0~1 之间的概率值
from sklearn.linear_model import LogisticRegression

# 评估指标：
#   accuracy_score: 准确率 = 预测对的 / 总预测数
#   classification_report: 一张表汇总精确率、召回率、F1
#   roc_auc_score: ROC 曲线下面积，衡量模型区分正负样本的能力（0.5=瞎猜，1=完美）
#   roc_curve: 画 ROC 曲线用的
#   confusion_matrix: 混淆矩阵，看模型把哪些样本分对了/分错了
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, roc_curve, confusion_matrix)

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# ==============================
# 第二步：加载数据
# ==============================
df = pd.read_csv(
    r'C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\data\interim\03_user_behavior_groups.csv'
)

print(f'原始数据量：{len(df):,} 行, {len(df.columns)} 列')
print(f'\n分组分布：')
print(df['group_type'].value_counts())
print(f'\n缺失率：')
for col in ['brand', 'category_code', 'category_id', 'price']:
    print(f'  {col}: {df[col].isna().mean()*100:.1f}%')

原始数据量：1,154,208 行, 10 列

分组分布：
group_type
B    518495
C    424063
A    211650
Name: count, dtype: int64

缺失率：
  brand: 44.1%
  category_code: 99.0%
  category_id: 0.0%
  price: 0.0%


In [3]:
# ==============================
# 第三步：数据清洗
# ==============================

# --- 3.1 排除 B 组 ---
# B 组（被动流失）= 加购后既没买也没移出，结果不确定
# 逻辑回归需要明确的 Y：买了(1) vs 没买(0)
# B 组是"悬而未决"的状态，放进去会污染模型
df = df[df['group_type'].isin(['A', 'C'])].copy()
print(f'排除 B 组后剩余：{len(df):,} 行')
print(f'A 组（购买）: {(df["group_type"]=="A").sum():,} ({(df["group_type"]=="A").mean()*100:.1f}%)')
print(f'C 组（流失）: {(df["group_type"]=="C").sum():,} ({(df["group_type"]=="C").mean()*100:.1f}%)')

# --- 3.2 创建目标变量 Y ---
# A=1（买了）, C=0（主动移除了）
# 这就是模型要预测的目标：给一条记录，预测它属于 1 还是 0
df['y_purchased'] = (df['group_type'] == 'A').astype(int)
print(f'\n目标变量分布：购买={df["y_purchased"].sum():,}, 流失={len(df)-df["y_purchased"].sum():,}')

# --- 3.3 处理价格异常值 ---
# 之前发现 price 有负值（最低 -79.37），可能是退款数据
# 先看看有多少
neg_price = (df['price'] < 0).sum()
print(f'\n负价格记录：{neg_price} 条')
# 去掉负价格和零价格（零价格没意义，可能是赠品或错误数据）
df = df[df['price'] > 0].copy()
print(f'清洗后剩余：{len(df):,} 行')

# --- 3.4 对价格做对数变换 ---
# 价格分布通常是右偏的（大量低价 + 少量高价），取对数后更接近正态分布
# 好处：让模型对"相对变化"敏感（从 1 涨到 2 vs 从 100 涨到 200 同等重要）
df['log_price'] = np.log(df['price'])

排除 B 组后剩余：635,713 行
A 组（购买）: 211,650 (33.3%)
C 组（流失）: 424,063 (66.7%)

目标变量分布：购买=211,650, 流失=424,063

负价格记录：17 条
清洗后剩余：635,362 行


In [4]:
# ==============================
# 第三点五步：设计首次加购前 Session 特征
# ==============================
# 背景：
# 之前的 08_session_features.csv 是按“整段 session”聚合出来的画像表。
# 问题是：整段 session 会包含第一次加购之后、甚至移出购物车之后的行为。
# 如果直接把这些字段放进模型，模型可能会偷看答案，导致分数虚高。

# 本次重新设计字段的目标：
# 预测时点 = 每个 user_session 的第一次 cart 事件发生时。
# 只使用预测时点之前已经发生的行为，也就是：event_time < first_cart_time。
# 这样更接近真实业务场景：用户第一次加购时，我们预测他后续会不会购买。

# 字段工程伪代码模板：
# 1. 读取原始事件明细 df_raw，并确保 event_time 是 datetime 类型。
#    df_raw['event_time'] = pd.to_datetime(df_raw['event_time'])

# 2. 找到每个 session 的开始时间。
#    session_start_time = df_raw.groupby('user_session')['event_time'].min()
#    含义：这个 session 第一条事件发生的时间。

# 3. 找到每个 session 的首次加购时间。
#    first_cart_time = df_raw[df_raw['event_type'] == 'cart'].groupby('user_session')['event_time'].min()
#    含义：这个 session 第一次出现 cart 的时间，也就是本模型的预测时点。
#    注意：没有 cart 的 session 没有首次加购时点，暂时不进入这版特征表。

# 4. 把 first_cart_time 合并回原始事件明细。
#    目的：让每一条事件都知道自己所在 session 的首次加购时间。
#    合并后才能判断这一条事件是在首次加购前，还是首次加购后。

# 5. 筛选首次加购前事件。
#    df_pre_cart = df_raw_with_first_cart[df_raw_with_first_cart['event_time'] < df_raw_with_first_cart['first_cart_time']]
#    注意：这里用 <，不用 <=，因为第一次 cart 本身不能算作“加购前行为”。

# 6. 按 user_session 聚合 df_pre_cart，生成首次加购前的 session 特征。
#    这些字段描述的是用户在第一次加购之前的浏览/比较状态。

# 第一版计划生成的字段：
# - pre_cart_has_view：首次加购前是否有 view 行为，有则 1，没有则 0
# - pre_cart_view_count：首次加购前 view 事件次数
# - pre_cart_unique_products：首次加购前接触过的不同 product_id 数
# - pre_cart_unique_brands：首次加购前接触过的不同 brand 数
# - pre_cart_unique_categories：首次加购前接触过的不同 category_id 数
# - pre_cart_avg_price：首次加购前事件涉及商品的平均 price
# - pre_cart_max_price：首次加购前事件涉及商品的最高 price
# - minutes_to_first_cart：session_start_time 到 first_cart_time 的分钟数

# 特殊情况处理：
# 如果某个 session 第一条事件就是 cart，那么它在首次加购前没有任何 view。
# 这类 session 不应该删除，因为它可能代表目标明确、老用户回购或外部入口直达。
# 对这类 session：pre_cart_has_view = 0，浏览/品牌/品类/价格类 pre_cart 字段可填 0。

# 粒度提醒：
# 当前建模主表是 user_session × product_id 粒度。
# pre_cart 特征是 user_session 粒度。
# 合并回主表后，同一个 session 下的多个商品记录会共享同一组 pre_cart 特征。
# 因此解释模型时，pre_cart 字段只能解释“用户当次 session 的购物状态”，不能解释成某个商品本身的原因。

In [5]:
# ==============================
# 第三点五步：实现首次加购前 Session 特征
# （设计思路见上方伪代码 cell，此处为可执行实现）
# ==============================

import psycopg2

# --- 3.5.1 从本地数据库读取原始事件明细 ---
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="",
    port=5432
)
df_raw = pd.read_sql("SELECT * FROM makeup_consumer_events.dec", conn)
conn.close()

df_raw['event_time'] = pd.to_datetime(df_raw['event_time'])
print(f'原始数据量：{len(df_raw):,} 行')
print(f'event_type 分布：')
print(df_raw['event_type'].value_counts())

# --- 3.5.2 找到每个 session 的首次加购时间 ---
# first_cart_time 是一个 Series，index = user_session，value = 该 session 第一次 cart 的时间
# 后面会用 map 把它挂回 df_raw，所以必须保证 index 是 user_session
first_cart_time = df_raw[df_raw['event_type'] == 'cart'].groupby('user_session')['event_time'].min()
print(f'\n有加购行为的 session 数：{len(first_cart_time):,}')

# --- 3.5.3 找到每个 session 的开始时间 ---
session_start_time = df_raw.groupby('user_session')['event_time'].min()

# --- 3.5.4 把 first_cart_time 合并回原始事件明细 ---
# 用 map 而不是 merge：map 是按 index 对齐一对一映射，比 merge 更轻量
# 前提条件：first_cart_time 的 index 必须是 user_session，
# 这样 df_raw['user_session'].map(first_cart_time) 才能把每行的 session 映射到对应的首次加购时间
df_raw_with_first_cart = df_raw.copy()
df_raw_with_first_cart['first_cart_time'] = df_raw_with_first_cart['user_session'].map(first_cart_time)
df_raw_with_first_cart['session_start_time'] = df_raw_with_first_cart['user_session'].map(session_start_time)

# 只保留有加购行为的 session（没有 cart 的 session 不进入特征表）
# 没有 cart 事件的 session 在 map 后 first_cart_time = NaN，dropna 直接剔除
df_raw_with_first_cart = df_raw_with_first_cart.dropna(subset=['first_cart_time'])
print(f'有加购 session 的事件总数：{len(df_raw_with_first_cart):,}')

# --- 3.5.5 篮选首次加购前事件（event_time < first_cart_time） ---
# 严格用 <，不包括首次 cart 本身：首次加购本身不算"加购前行为"
df_pre_cart = df_raw_with_first_cart[
    df_raw_with_first_cart['event_time'] < df_raw_with_first_cart['first_cart_time']
].copy()
print(f'首次加购前事件数：{len(df_pre_cart):,}')

# --- 3.5.6 按 session 聚合生成 pre-cart 特征 ---
# 口径说明：所有聚合字段都只基于 event_type == 'view' 的事件
# 即只看用户在首次加购前"浏览"了什么，不看 remove_from_cart 等其他行为
# 这是因为首次加购前唯一可能的行为就是 view（还没有 cart/remove/purchase）
pre_cart_view_agg = df_pre_cart[df_pre_cart['event_type'] == 'view'].groupby('user_session').agg(
    pre_cart_view_count=('event_type', 'count'),              # 首次加购前 view 事件的总次数
    pre_cart_unique_products=('product_id', 'nunique'),       # 首次加购前 view 过的不同商品数
    pre_cart_unique_brands=('brand', 'nunique'),              # 首次加购前 view 过的不同品牌数
    pre_cart_unique_categories=('category_id', 'nunique'),    # 首次加购前 view 过的不同品类数
    pre_cart_avg_price=('price', 'mean'),                     # 首次加购前 view 事件对应商品的平均价格
    pre_cart_max_price=('price', 'max')                       # 首次加购前 view 事件对应商品的最高价格
)

# 所有有加购行为的 session 列表（作为特征表的骨架）
all_cart_sessions = first_cart_time.index

# 构建完整特征表骨架
pre_cart_features = pd.DataFrame({'user_session': all_cart_sessions})

# 合入 view 聚合结果（没有 pre-cart view 的 session 会得到 NaN，后面统一填 0）
pre_cart_features = pre_cart_features.merge(
    pre_cart_view_agg.reset_index(),
    on='user_session',
    how='left'
)

# --- 3.5.7 处理特殊情况：没有浏览直接加购的 session ---
# 这类 session 可能代表目标明确、老用户回购或外部入口直达，不应删除
# pre_cart_has_view 标记：1=有浏览再加购，0=直接加购无浏览
pre_cart_features['pre_cart_has_view'] = (
    pre_cart_features['pre_cart_view_count'].notna() & (pre_cart_features['pre_cart_view_count'] > 0)
).astype(int)
# 其余浏览类字段统一填 0（NaN 代表该 session 在加购前没有任何 view 行为）
pre_cart_features['pre_cart_view_count'] = pre_cart_features['pre_cart_view_count'].fillna(0).astype(int)
pre_cart_features['pre_cart_unique_products'] = pre_cart_features['pre_cart_unique_products'].fillna(0).astype(int)
pre_cart_features['pre_cart_unique_brands'] = pre_cart_features['pre_cart_unique_brands'].fillna(0).astype(int)
pre_cart_features['pre_cart_unique_categories'] = pre_cart_features['pre_cart_unique_categories'].fillna(0).astype(int)
pre_cart_features['pre_cart_avg_price'] = pre_cart_features['pre_cart_avg_price'].fillna(0)
pre_cart_features['pre_cart_max_price'] = pre_cart_features['pre_cart_max_price'].fillna(0)

# --- 3.5.8 计算 minutes_to_first_cart ---
# 口径：session 开始时间 → 首次加购时间 的分钟数
# 衡量用户从进入 session 到做出第一次加购决策花了多久
minutes_to_first_cart_map = (first_cart_time - session_start_time).dt.total_seconds() / 60
pre_cart_features['minutes_to_first_cart'] = pre_cart_features['user_session'].map(minutes_to_first_cart_map)

print(f'\nPre-cart 特征表：{len(pre_cart_features):,} 行, {len(pre_cart_features.columns)} 列')
print(f'\n特征概览：')
print(pre_cart_features.describe().round(2))
print(f'\npre_cart_has_view 分布（0=无浏览直接加购, 1=有浏览再加购）：')
print(pre_cart_features['pre_cart_has_view'].value_counts())

# --- 3.5.9 保存特征表 ---
pre_cart_features.to_csv(
    r'C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\data\interim\09_pre_cart_features.csv',
    index=False
)
print('\n✅ 特征表已保存至 data/interim/09_pre_cart_features.csv')

原始数据量：3,533,286 行
event_type 分布：
event_type
view                1728331
cart                 927124
remove_from_cart     664655
purchase             213176
Name: count, dtype: int64

有加购行为的 session 数：165,296
有加购 session 的事件总数：2,398,758
首次加购前事件数：332,232

Pre-cart 特征表：165,296 行, 9 列

特征概览：
       pre_cart_view_count  pre_cart_unique_products  pre_cart_unique_brands  \
count            165296.00                 165296.00               165296.00   
mean                  1.26                      1.11                    0.77   
std                   2.61                      2.11                    0.93   
min                   0.00                      0.00                    0.00   
25%                   0.00                      0.00                    0.00   
50%                   1.00                      1.00                    1.00   
75%                   1.00                      1.00                    1.00   
max                 105.00                     91.00                   

In [6]:
# ==============================
# 第四步：特征工程
# ==============================
# 特征工程 = 从原始数据中提取/构造对预测有用的变量
# 这一步决定了模型能学到什么，是整个建模中最关键的环节

# --- 4.1 时间特征 ---
df['event_time'] = pd.to_datetime(df['event_time'])

# 小时（0-23）：之前分析发现上午购买率高，晚上流失率高
df['hour'] = df['event_time'].dt.hour

# 星期几（0=周一, 6=周日）：看看工作日和周末有没有差异
df['day_of_week'] = df['event_time'].dt.dayofweek

# --- 4.2 品牌特征 ---
# brand 缺失 44%，但"缺失"本身可能是一个信号
# 比如：没有品牌的商品可能更容易被比价、更容易流失
df['brand_missing'] = df['brand'].isna().astype(int)

# 品牌太多太散，直接用会维度爆炸
# 策略：先把 NaN 填成 'no_brand'，再做 Top N，其余归为 "other"
# 为什么选 15？前 15 个品牌覆盖了大部分数据，
# 太少会丢信息，太多会产生很多稀疏特征

# 【关键顺序】必须先 fillna，再做 Top N 映射！
# 如果反过来（先 apply 再 fillna），NaN 会在 apply 阶段被映射成 'other'，
# 后面的 fillna 就找不到 NaN 了
TOP_N_BRANDS = 15
df['brand_clean'] = df['brand'].fillna('no_brand')  # 第一步：先把 NaN 标记出来
top_brands = df['brand_clean'].value_counts().head(TOP_N_BRANDS).index.tolist()
df['brand_clean'] = df['brand_clean'].apply(         # 第二步：再把小众品牌归为 other
    lambda x: x if x in top_brands else 'other'
)

print(f'品牌处理后分布（Top 15 + other + no_brand）:')
print(df['brand_clean'].value_counts())

# --- 4.3 品类特征 ---
# category_code 缺失 99%，废了，用 category_id 代替
# 同样做 Top N 处理（category_id 没有缺失，不需要 fillna）
TOP_N_CATS = 15
top_cats = df['category_id'].value_counts().head(TOP_N_CATS).index.tolist()
df['cat_clean'] = df['category_id'].apply(
    lambda x: x if x in top_cats else 'other'
)

print(f'\n品类处理后分布:')
print(df['cat_clean'].value_counts())

# --- 4.4 合并首次加购前 Session 特征 ---
# pre_cart_features 是 user_session 粒度；df 是 user_session × product_id 粒度。
# 合并后，同一个 session 下的多个商品记录会共享同一组 pre-cart 特征。
pre_cart_cols = [
    'pre_cart_has_view',
    'pre_cart_view_count',
    'pre_cart_unique_products',
    'pre_cart_unique_brands',
    'pre_cart_unique_categories',
    'pre_cart_avg_price',
    'pre_cart_max_price',
    'minutes_to_first_cart',
]

df = df.merge(pre_cart_features[['user_session'] + pre_cart_cols], on='user_session', how='left')

# 理论上 A/C 组都来自加购相关行为，大部分都应匹配到 pre_cart 特征。
# 少量匹配不到的记录统一填 0，避免后续建模报 NaN。
match_rate = df['minutes_to_first_cart'].notna().mean()
df[pre_cart_cols] = df[pre_cart_cols].fillna(0)

print(f'\nPre-cart 特征合并完成，匹配率：{match_rate*100:.1f}%')
print(df[pre_cart_cols].describe().round(2))

品牌处理后分布（Top 15 + other + no_brand）:
brand_clean
no_brand     279248
other        126635
runail        49419
irisk         29324
masura        28188
grattol       25496
bpw.style     18540
ingarden      13620
estel         10510
pole          10201
freedecor      8098
kapous         7918
uno            7774
bluesky        7561
haruyama       6607
milv           6223
Name: count, dtype: int64

品类处理后分布:
cat_clean
other                  374120
1487580007675986893     47577
1487580005595612013     26973
1487580005671109489     24296
1487580006317032337     22754
1487580005092295511     21193
1487580005754995573     19114
1602943681873052386     18247
1487580005268456287     13614
1487580005134238553     12071
1487580009286598681     11515
1487580008145748965      8909
1487580005511725929      8886
1487580010100293687      8739
1487580009445982239      8702
1487580008246412266      8652
Name: count, dtype: int64

Pre-cart 特征合并完成，匹配率：82.9%
       pre_cart_has_view  pre_cart_view_count  pre_ca

In [7]:
# ==============================
# 第五步：编码 + 拆分训练集/测试集
# ==============================

# --- 5.1 选出要用到的特征列 ---

# 数值型特征：直接喂给模型（后面会标准化）
# 原来的 3 个 + 新增的 session 维度特征
numeric_features = [
    'log_price', 'hour', 'day_of_week',           # 原来的：商品属性 + 时间
    # --- 新增：session 维度 ---
    'pre_cart_view_count',          # 首次加购前 view 事件次数
    'pre_cart_unique_products',     # 首次加购前浏览过的不同商品数
    'pre_cart_unique_brands',       # 首次加购前浏览过的不同品牌数
    'pre_cart_unique_categories',   # 首次加购前浏览过的不同品类数
    'pre_cart_avg_price',           # 首次加购前浏览商品的平均价格
    'pre_cart_max_price',           # 首次加购前浏览商品的最高价格
    'minutes_to_first_cart'         # session 开始到第一次加购花了多少分钟
]

# 分类型特征：需要做 one-hot 编码
categorical_features = ['brand_clean', 'cat_clean']

# 二值特征：已经是 0/1，不需要额外处理
binary_features = ['brand_missing', 'pre_cart_has_view']

# 合并所有特征
feature_cols = numeric_features + categorical_features + binary_features
print(f'使用的特征数：{len(feature_cols)} 个')
print(f'  数值型({len(numeric_features)}个)：{numeric_features}')
print(f'  分类型({len(categorical_features)}个)：{categorical_features}')
print(f'  二值型({len(binary_features)}个)：{binary_features}')

# --- 5.2 One-Hot 编码 ---
# One-Hot 编码：把分类变量转成多个 0/1 列
# 比如 brand_clean 有 17 个值 -> 变成 17 列，每列是 0 或 1
# 例：brand_clean=runail -> brand_runail=1, 其余=0
#

# drop_first=True：丢掉第一列，避免"多重共线性"
#   原因：如果 17 列里有 16 列都是 0，那第 17 列必然是 1
#   这种"可以用其他列推导出来"的情况叫共线性，会让系数不稳定
#   丢掉一列后，被丢的那个品牌就变成了"参照基准"
df_model = pd.get_dummies(
    df[feature_cols + ['y_purchased']],
    columns=categorical_features,
    drop_first=True,
    dtype=int  # 确保生成的列是 int 而非 bool
)


# 看看编码后有多少列
X_cols = [c for c in df_model.columns if c != 'y_purchased']
print(f'\n编码后特征数：{len(X_cols)} 列')
print(f'特征列名：{X_cols}')

# --- 5.3 拆分 X 和 Y ---
X = df_model[X_cols]
y = df_model['y_purchased']

# --- 5.4 拆分训练集和测试集 ---
# test_size=0.2：80% 训练，20% 测试
# random_state=42：固定随机种子，保证每次拆分一样（可复现）
# stratify=y：按 Y 的比例拆分，保证训练集和测试集中 A/C 的比例一致
#   如果不加这个参数，可能出现测试集里全是 A 或全是 C 的极端情况
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\n训练集：{len(X_train):,} 行（购买率 {y_train.mean()*100:.1f}%）')
print(f'测试集：{len(X_test):,} 行（购买率 {y_test.mean()*100:.1f}%）')

# --- 5.5 标准化 ---
# 只对数值型特征做标准化（分类型已经是 0/1，不需要）
# fit_transform：在训练集上计算均值和标准差，然后变换
# transform：在测试集上只变换（用训练集的均值和标准差，不能重新算）
#   为什么？因为测试集模拟"未来的新数据"，我们不能用未来数据的信息
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print(f'\n标准化完成，共标准化了 {len(numeric_features)} 个数值特征')

使用的特征数：14 个
  数值型(10个)：['log_price', 'hour', 'day_of_week', 'pre_cart_view_count', 'pre_cart_unique_products', 'pre_cart_unique_brands', 'pre_cart_unique_categories', 'pre_cart_avg_price', 'pre_cart_max_price', 'minutes_to_first_cart']
  分类型(2个)：['brand_clean', 'cat_clean']
  二值型(2个)：['brand_missing', 'pre_cart_has_view']

编码后特征数：42 列
特征列名：['log_price', 'hour', 'day_of_week', 'pre_cart_view_count', 'pre_cart_unique_products', 'pre_cart_unique_brands', 'pre_cart_unique_categories', 'pre_cart_avg_price', 'pre_cart_max_price', 'minutes_to_first_cart', 'brand_missing', 'pre_cart_has_view', 'brand_clean_bpw.style', 'brand_clean_estel', 'brand_clean_freedecor', 'brand_clean_grattol', 'brand_clean_haruyama', 'brand_clean_ingarden', 'brand_clean_irisk', 'brand_clean_kapous', 'brand_clean_masura', 'brand_clean_milv', 'brand_clean_no_brand', 'brand_clean_other', 'brand_clean_pole', 'brand_clean_runail', 'brand_clean_uno', 'cat_clean_1487580005134238553', 'cat_clean_1487580005268456287', 'cat_cle

In [11]:
# ==============================
# 第六步：训练逻辑回归模型
# ==============================

# LogisticRegression 参数说明：
#   max_iter=1000: 最大迭代次数（默认 100 有时不够用，会报收敛警告）
#   solver='lbfgs': 优化算法（默认值，适合中小规模数据）
#   penalty='l2': L2 正则化（防止过拟合，给系数加一个"不要太大"的约束）
#   C=1.0: 正则化强度的倒数（C 越小 -> 正则化越强 -> 系数越小 -> 更保守）
#          1.0 是默认值，后续可以通过调参优化
#   random_state=42: 固定随机种子
model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    penalty='l2',
    C=1.0,
    random_state=42,
    class_weight='balanced',
)

# fit = 训练（让模型在训练集上学习特征和 Y 之间的关系）
model.fit(X_train_scaled, y_train)

print('模型训练完成!')
print(f'特征数: {len(X_cols)}')
print(f'训练样本数: {len(X_train):,}')

模型训练完成!
特征数: 42
训练样本数: 508,289


In [13]:
# ==============================
# 第七步：评估模型
# ==============================


y_prob = model.predict_proba(X_test_scaled)[:, 1]  # 取"购买"的概率（第 1 列）
# --- 7.1 在测试集上做预测 ---
# predict_proba: 返回购买概率
# 默认 model.predict() 等价于 threshold=0.5
# 这里手动设置阈值，观察购买类 precision / recall 的变化
y_prob = model.predict_proba(X_test_scaled)[:, 1]

threshold = 0.5
y_pred = (y_prob >= threshold).astype(int)

print(f'当前分类阈值 threshold = {threshold}')

# --- 7.2 准确率 ---
acc = accuracy_score(y_test, y_pred)
print(f'准确率 (Accuracy): {acc:.4f}')
print(f'  -> 含义：模型预测对了 {acc*100:.1f}% 的样本')

# --- 7.3 分类报告 ---
# 比准确率更有用，因为它分别看了每个类别的表现：
#   Precision（精确率）= 预测为正的里面，真正为正的比例 -> "预测购买的用户中，多少真的买了"
#   Recall（召回率）= 真正为正的里面，被预测出来的比例 -> "真正购买的用户中，多少被模型抓到了"
#   F1 = Precision 和 Recall 的调和平均，综合衡量
print(f'\n分类报告:')
print(classification_report(y_test, y_pred, target_names=['流失(0)', '购买(1)']))

# --- 7.4 ROC-AUC ---
# ROC 曲线：横轴是假阳性率（误报），纵轴是真阳性率（命中率）
# AUC 值：曲线下面积，0.5=瞎猜，1=完美
#   0.5-0.6: 基本没用
#   0.6-0.7: 一般
#   0.7-0.8: 还行
#   0.8-0.9: 不错
#   0.9+:    很好
auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')
print(f'  -> 含义：随机取一个购买用户和一个流失用户，模型给购买用户打更高分的概率是 {auc*100:.1f}%')
print(f'  -> 之前无 session 特征时 AUC = 0.5707')

# --- 7.5 混淆矩阵 ---
# 四个格子：
#   TN(真负) = 实际流失，预测流失 -> 正确
#   FP(假正) = 实际流失，预测购买 -> 误报
#   FN(假负) = 实际购买，预测流失 -> 漏报
#   TP(真正) = 实际购买，预测购买 -> 正确
cm = confusion_matrix(y_test, y_pred)
print(f'\n混淆矩阵:')
print(f'                    预测流失    预测购买')
print(f'  实际流失     {cm[0][0]:>8,}  {cm[0][1]:>8,}')
print(f'  实际购买     {cm[1][0]:>8,}  {cm[1][1]:>8,}')

当前分类阈值 threshold = 0.5
准确率 (Accuracy): 0.5370
  -> 含义：模型预测对了 53.7% 的样本

分类报告:
              precision    recall  f1-score   support

       流失(0)       0.72      0.50      0.59     84764
       购买(1)       0.38      0.60      0.46     42309

    accuracy                           0.54    127073
   macro avg       0.55      0.55      0.53    127073
weighted avg       0.60      0.54      0.55    127073

ROC-AUC: 0.5773
  -> 含义：随机取一个购买用户和一个流失用户，模型给购买用户打更高分的概率是 57.7%
  -> 之前无 session 特征时 AUC = 0.5707

混淆矩阵:
                    预测流失    预测购买
  实际流失       42,766    41,998
  实际购买       16,842    25,467


In [10]:
# ==============================
# 第八步：解读模型系数（最重要的环节）
# ==============================
# 逻辑回归的最大优点：系数可以直接解释！
#
# 系数 (coef) 的含义：
#   正值 -> 该特征增加了购买的概率
#   负值 -> 该特征降低了购买的概率（增加了流失风险）
#
# 优势比 OR (Odds Ratio) = e^系数：
#   OR > 1 -> 该特征让购买的可能性增加了多少倍
#   OR < 1 -> 该特征让购买的可能性降低了多少倍
#   OR = 1 -> 没有影响
#   例：OR=1.5 表示购买的可能性提高了 50%
#   例：OR=0.7 表示购买的可能性降低了 30%（1-0.7=0.3）

# 提取系数
coef_df = pd.DataFrame({
    '特征': X_cols,
    '系数': model.coef_[0],
    'OR(优势比)': np.exp(model.coef_[0])
})

# 按系数从大到小排序
coef_df = coef_df.sort_values('系数', ascending=False).reset_index(drop=True)

# 加上截距（基准线）
intercept = model.intercept_[0]
print(f'截距 (Intercept): {intercept:.4f}')
print(f'  -> 含义：当所有特征都为 0 时的基准 log-odds')

print(f'\n各特征的系数和优势比：')
print('=' * 70)
for _, row in coef_df.iterrows():
    direction = '↑ 促进购买' if row['系数'] > 0 else '↓ 促进流失'
    print(f"{row['特征']:<35} 系数={row['系数']:>7.3f}  OR={row['OR(优势比)']:>6.3f}  {direction}")

截距 (Intercept): -1.0514
  -> 含义：当所有特征都为 0 时的基准 log-odds

各特征的系数和优势比：
brand_clean_bpw.style               系数=  0.612  OR= 1.845  ↑ 促进购买
brand_clean_estel                   系数=  0.561  OR= 1.753  ↑ 促进购买
brand_clean_kapous                  系数=  0.547  OR= 1.729  ↑ 促进购买
brand_clean_uno                     系数=  0.481  OR= 1.617  ↑ 促进购买
brand_clean_runail                  系数=  0.463  OR= 1.589  ↑ 促进购买
cat_clean_1487580009286598681       系数=  0.437  OR= 1.548  ↑ 促进购买
brand_clean_freedecor               系数=  0.378  OR= 1.459  ↑ 促进购买
brand_clean_other                   系数=  0.375  OR= 1.455  ↑ 促进购买
brand_clean_milv                    系数=  0.358  OR= 1.431  ↑ 促进购买
cat_clean_1487580009445982239       系数=  0.312  OR= 1.366  ↑ 促进购买
brand_clean_irisk                   系数=  0.279  OR= 1.322  ↑ 促进购买
cat_clean_1487580010100293687       系数=  0.250  OR= 1.284  ↑ 促进购买
brand_clean_grattol                 系数=  0.213  OR= 1.238  ↑ 促进购买
cat_clean_1487580005268456287       系数=  0.192  OR= 1.211  ↑ 促进购买
brand_c